### GA4 Ecommerce EDA
Loading the 8 tables I exported from Bigquery. 
Need to make sure the numbers match with SQl, if not, back to SQL 


In [1]:
import pandas as pd
import numpy as np

# Just setting this so I don't get scientific notation everywhere
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

# file location
DATA_DIR = '../docs/'


files = {
    'channel': 'm_channel_performance.csv',
    'geo': 'm_daily_pattern_by_geo.csv',
    'activation': 'm_first_visit_activation.csv',
    'funnel': 'm_funnel_by_device_channel.csv',
    'retention': 'm_retention_curve.csv',
    'visits': 'm_visits_before_purchase.csv',
    'category': 'm_category_performance_geo_device.csv',
    'revenue': 'm_revenue_distribution.csv',
}

# Load 
dfs = {}
for name, f in files.items():
    dfs[name] = pd.read_csv(DATA_DIR + f)
    print(f"{name:15} loaded. Shape: {dfs[name].shape}")

channel         loaded. Shape: (5, 4)
geo             loaded. Shape: (35, 5)
activation      loaded. Shape: (1, 3)
funnel          loaded. Shape: (15, 5)
retention       loaded. Shape: (14, 6)
visits          loaded. Shape: (1, 2)
category        loaded. Shape: (1534, 5)
revenue         loaded. Shape: (1, 5)


 ##### --- 1. Quick Validation ---

 Just making sure the export is fine.


In [7]:
# 1. Channel
df = dfs['channel']
assert len(df) == 5, "Wait, why isn't this 5 rows?"
print("Channel check: OK.")

# 2. Geo
df = dfs['geo']
assert len(df) == 35, "Expected 35 rows (7 days * 5 groups)"
print("Geo check: OK.")

# 3. Activation
df = dfs['activation']
# Just checking if the rate is between 0 and 1. If it's > 1, something is broken.
assert 0 <= df['view_to_cart_rate_first_day'].iloc[0] <= 1
print("Activation check: OK.")

# 4. Funnel
df = dfs['funnel']
# If purchase > cart, the query has some issue.
violations = df[(df['purchase_users'] > df['add_to_cart_users']) | df['add_to_cart_users'] > df['view_item_users'] ]
assert violations.empty, "Funnel logic is broken somewhere."
print("Funnel check: OK.")

# 5. Retention
df = dfs['retention']
df['cohort_week'] = pd.to_datetime(df['cohort_week'])
# Just making sure dates look right.
assert df['cohort_week'].isna().sum() == 0, "Null cohort_week found; stale table??"
assert df['cohort_week'].min() >= pd.Timestamp('2020-11-01')
assert df['cohort_week'].max() <= pd.Timestamp('2021-01-31')
print("Retention check: OK.")

# 6. Visits
df = dfs['visits']
assert 0 < df['avg_active_days_before_purchase'].iloc[0] < 30
print("Visits check: OK.")

# 7. Category
# This one had that weird 'Unknown' bucket issue
df = dfs['category']
total_rev = df['total_revenue'].sum()
print(f"Total revenue: {total_rev:,.2f}")
# Allowing a small margin of error because of rounding in SQL vs Pandas
assert abs(total_rev - 362110) < 500 

# check no junk category labels leaked through
junk = df[df['category_clean'].isin(['(not set)', '']) | df['category_clean'].isna()]
assert junk.empty, f"Junk category values leaked through:\n{junk}"

## confirm nulls are still confined to Unknown only (43 in notebook vs 44 as earlier seen in raw csv inspeaction)
nulls = df[df['total_revenue'].isna()]
print(nulls['category_clean'].value_counts())
assert (nulls['category_clean'] == 'Unknown').all(), "Nulls found outside Unknown category!"
print("Category check: OK.")

# 8. Revenue Dist
df = dfs['revenue']
assert len(df) == 1
assert abs(df['median_rev'].iloc[0] - 48.0) < 1, "Median does not match finding"
assert abs(df['mean_rev'].iloc[0] - 69.09) < 1, "Mean doesn't match exploration finding"
print("Revenue dist check: OK.")

Channel check: OK.
Geo check: OK.
Activation check: OK.
Funnel check: OK.
Retention check: OK.
Visits check: OK.
Total revenue: 362,110.00
category_clean
Unknown    43
Name: count, dtype: int64
Category check: OK.
Revenue dist check: OK.


#### --- 2. EDA Stuff ---


In [8]:
# --- 2. EDA Stuff ---

# Let's see what we're working with
dfs['channel'].describe()

# Category performance is the big one. Let's look at the top categories.
# Man, I really need to clean these names up properly later.
dfs['category']['category_clean'].value_counts().head(10)

# Total revenue by country. Does this look right?
# (Self-note: Need to check if 'United States' is at the top, it usually is)
dfs['category'].groupby('country_clean')['total_revenue'].sum().sort_values(ascending=False).head(5)

# Pivot table for the report later
pivot_df = dfs['category'].pivot_table(
    index='category_clean',
    columns='device_category',
    values='total_revenue',
    aggfunc='sum',
    fill_value=0
)
pivot_df.sort_values('desktop', ascending=False).head(10)

device_category,desktop,mobile,tablet
category_clean,,,
Apparel,"98,030.00","70,057.00","3,640.00"
New,"14,865.00","10,554.00",394.00
Bags,"13,197.00","10,115.00",548.00
Campus Collection,"11,629.00","7,996.00",436.00
Accessories,"11,067.00","6,484.00",264.00
Uncategorized Items,"10,586.00","6,329.00",479.00
Shop by Brand,"10,451.00","6,407.00",102.00
Drinkware,"8,970.00","6,553.00",284.00
Lifestyle,"8,343.00","4,975.00",105.00


##### --- 3. Nulls ---
Checking for nulls. 
Note that 'category_performance' has some nulls in revenue, but that's expected for the 'Unknown' category.

In [9]:
for name, df in dfs.items():
    if df.isna().sum().sum() > 0:
        print(f"Nulls found in {name}:")
        print(df.isna().sum()[df.isna().sum() > 0])
        print("-" * 20)

Nulls found in category:
total_revenue    43
dtype: int64
--------------------


##### --- 4. Stats ---
 Right-skewed revenue. Standard ecommerce stuff.

In [10]:
# Right-skewed revenue. Standard ecommerce stuff.
rev = dfs['revenue'].iloc[0]
print(f"Mean: {rev['mean_rev']:.2f}")
print(f"Median: {rev['median_rev']:.2f}")
# If mean > median, it's right-skewed. 
# Definitely use median for the presentation. Stakeholders always get confused by means.

Mean: 69.09
Median: 48.00


##### TODO:
 - Need to build the visualizations next. 
 - Bar charts for channels.
 - Line chart for retention.
 - Maybe a heatmap for the day of week stuff?

In [11]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

##### -- Bar chart (channel conversion rate) --